In [21]:
import os
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 프로젝트 경로 설정 (본인의 실제 경로로 수정하세요)
project_path = '/content/drive/MyDrive/cloud-computing/cloud-computing-term-project'

# 3. 경로 이동 및 확인
if os.path.exists(project_path):
    %cd {project_path}
    !git status  # 현재 브랜치와 상태 확인
else:
    print("경로를 찾을 수 없습니다. 드라이브 내 폴더 명을 확인해주세요.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/cloud-computing/cloud-computing-term-project
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [ ]:
# 필요한 패키지 설치
!pip install -q openai-whisper groq edge-tts gtts pydub soundfile
!pip install -q ffmpeg-python
!apt-get install -qq ffmpeg

print('✅ 모든 패키지 설치 완료!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
✅ 모든 패키지 설치 완료!


In [ ]:
import os

# ── 방법 A: Google Colab Secrets 사용 (권장) ──────────────────────────
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print('✅ Colab Secrets에서 API 키를 불러왔습니다.')
except Exception:
    # ── 방법 B: 직접 입력 ──────────────────────────────────────────────
    GROQ_API_KEY = 'gsk_여기에_Groq_API_키를_입력하세요'
    print('⚠️  API 키를 직접 입력했습니다. Secrets 사용을 권장합니다.')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print(f'🔑 API 키 설정 완료: {GROQ_API_KEY[:10]}...')

✅ Colab Secrets에서 API 키를 불러왔습니다.
🔑 API 키 설정 완료: gsk_tzgRGd...


In [ ]:
import whisper
import groq
import edge_tts
import asyncio
import json
import time
from pathlib import Path
from IPython.display import Audio, display, HTML
import warnings
warnings.filterwarnings('ignore')

# ── Whisper 모델 로드 (base: 빠름, small: 권장, medium: 정확) ──────────
print('🔄 Whisper 모델 로딩 중... (첫 실행 시 다운로드 필요)')
whisper_model = whisper.load_model('base')  # 'small' 도 추천
print('✅ Whisper 모델 로드 완료!')

# ── Groq 클라이언트 초기화 ─────────────────────────────────────────────
groq_client = groq.Groq(api_key=GROQ_API_KEY)
print('✅ Groq 클라이언트 초기화 완료!')

🔄 Whisper 모델 로딩 중... (첫 실행 시 다운로드 필요)


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 280MiB/s]


✅ Whisper 모델 로드 완료!
✅ Groq 클라이언트 초기화 완료!


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STT: 음성 → 텍스트 (Whisper)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def transcribe_audio(audio_path: str) -> dict:
    """
    Whisper로 음성 파일을 텍스트로 변환합니다.
    Returns: {'text': str, 'language': str, 'duration': float}
    """
    print(f'🎙️  STT 처리 중: {audio_path}')
    start = time.time()

    result = whisper_model.transcribe(
        audio_path,
        language='en',        # 영어 강제 (None 으로 바꾸면 자동 감지)
        task='transcribe',
        fp16=False             # CPU 환경에서는 False
    )

    elapsed = time.time() - start
    text = result['text'].strip()
    lang = result.get('language', 'en')

    print(f'   📝 인식된 텍스트: "{text}"')
    print(f'   🌐 감지된 언어: {lang}')
    print(f'   ⏱️  처리 시간: {elapsed:.2f}초')

    return {'text': text, 'language': lang, 'duration': elapsed}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  LLM: 텍스트 교정 (Groq + Llama 3.3)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CORRECTION_SYSTEM_PROMPT = """
You are an expert English conversation coach specializing in helping non-native speakers improve their spoken English.

Your task is to analyze the user's spoken English and provide detailed, encouraging feedback.

Always respond in the following JSON format:
{
  "original": "<the original text as spoken>",
  "corrected": "<the grammatically correct and natural version>",
  "is_correct": <true if no corrections needed, false otherwise>,
  "errors": [
    {
      "type": "<grammar|vocabulary|pronunciation_hint|fluency>",
      "original": "<the incorrect part>",
      "corrected": "<the correct version>",
      "explanation": "<clear explanation in Korean>"
    }
  ],
  "overall_feedback": "<encouraging overall feedback in Korean, 2-3 sentences>",
  "better_expressions": ["<alternative natural expression 1>", "<alternative natural expression 2>"],
  "pronunciation_tips": "<specific pronunciation advice in Korean if needed>",
  "score": <naturalness score 1-10>
}

Be specific, educational, and encouraging. Focus on the most important improvements.
"""

def correct_english(text: str, context: str = "") -> dict:
    """
    Groq LLM으로 영어 텍스트를 교정합니다.
    Returns: 교정 결과 딕셔너리
    """
    print('\n🤖 LLM 교정 중...')
    start = time.time()

    user_message = f"Please correct and analyze this spoken English: \"{text}\""
    if context:
        user_message += f"\nConversation context: {context}"

    response = groq_client.chat.completions.create(
        model='llama-3.3-70b-versatile',   # 무료 티어에서 사용 가능한 강력한 모델
        messages=[
            {'role': 'system', 'content': CORRECTION_SYSTEM_PROMPT},
            {'role': 'user',   'content': user_message}
        ],
        temperature=0.3,
        max_tokens=1024,
        response_format={'type': 'json_object'}  # JSON 출력 강제
    )

    elapsed = time.time() - start
    raw = response.choices[0].message.content
    result = json.loads(raw)

    print(f'   ⏱️  처리 시간: {elapsed:.2f}초')
    print(f'   📊 자연스러움 점수: {result.get("score", "N/A")}/10')

    return result


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  TTS: 텍스트 → 음성 (Edge-TTS: Microsoft 무료)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
async def text_to_speech_async(
    text: str,
    output_path: str = '/content/output.mp3',
    voice: str = 'en-US-JennyNeural',    # 자연스러운 미국 영어 여성 음성
    rate: str = '+0%',                    # 속도 조절 (-50% ~ +50%)
    pitch: str = '+0Hz'
) -> str:
    """Edge-TTS로 텍스트를 음성으로 변환합니다."""
    communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
    await communicate.save(output_path)
    return output_path


def text_to_speech(
    text: str,
    output_path: str = '/content/output.mp3',
    voice: str = 'en-US-JennyNeural'
) -> str:
    """동기 래퍼 함수 — Colab 호환 버전"""
    print(f'\n🔊 TTS 생성 중: "{text[:50]}..."' if len(text) > 50 else f'\n🔊 TTS 생성 중: "{text}"')
    start = time.time()

    loop = asyncio.get_event_loop()
    loop.run_until_complete(text_to_speech_async(text, output_path, voice))

    elapsed = time.time() - start
    print(f'   ✅ 음성 파일 저장: {output_path}')
    print(f'   ⏱️  처리 시간: {elapsed:.2f}초')
    return output_path


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  결과 출력 헬퍼
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def display_correction_result(result: dict):
    """교정 결과를 보기 좋게 출력합니다."""
    score = result.get('score', 0)
    score_emoji = '🟢' if score >= 8 else ('🟡' if score >= 5 else '🔴')

    html = f"""
    <div style="font-family: 'Noto Sans KR', sans-serif; max-width: 700px; margin: 10px 0;">

      <div style="background: #1e1e2e; color: #cdd6f4; border-radius: 12px; padding: 20px; margin-bottom: 12px;">
        <h3 style="color: #cba6f7; margin: 0 0 12px;">📊 교정 결과 {score_emoji} {score}/10</h3>

        <p><b style="color: #f38ba8;">원문:</b> {result.get('original', '')}</p>
        <p><b style="color: #a6e3a1;">교정:</b> {result.get('corrected', '')}</p>

        {'<p style="color:#a6e3a1;">✅ 완벽한 문장입니다!</p>' if result.get('is_correct') else ''}
      </div>
    """

    # 오류 목록
    errors = result.get('errors', [])
    if errors:
        html += '<div style="background: #313244; border-radius: 12px; padding: 20px; margin-bottom: 12px;">'
        html += '<h4 style="color: #fab387; margin: 0 0 10px;">🔍 오류 분석</h4>'
        for err in errors:
            type_colors = {'grammar': '#f38ba8', 'vocabulary': '#fab387',
                           'pronunciation_hint': '#89dceb', 'fluency': '#a6e3a1'}
            c = type_colors.get(err.get('type', ''), '#cdd6f4')
            html += f"""
            <div style="border-left: 3px solid {c}; padding: 8px 12px; margin: 8px 0; background: #1e1e2e; border-radius: 4px;">
              <span style="color:{c}; font-size:12px; font-weight:bold;">[{err.get('type','').upper()}]</span><br>
              ❌ <code style="color:#f38ba8;">{err.get('original','')}</code>
              → ✅ <code style="color:#a6e3a1;">{err.get('corrected','')}</code><br>
              <small style="color:#bac2de;">{err.get('explanation','')}</small>
            </div>"""
        html += '</div>'

    # 더 나은 표현
    better = result.get('better_expressions', [])
    if better:
        html += '<div style="background: #313244; border-radius: 12px; padding: 20px; margin-bottom: 12px;">'
        html += '<h4 style="color: #89b4fa; margin: 0 0 10px;">💡 더 자연스러운 표현</h4>'
        for expr in better:
            html += f'<div style="background: #1e1e2e; padding: 8px 12px; margin: 6px 0; border-radius: 6px; color: #89dceb;">🗣️ {expr}</div>'
        html += '</div>'

    # 종합 피드백
    html += f"""
      <div style="background: #313244; border-radius: 12px; padding: 20px;">
        <h4 style="color: #a6e3a1; margin: 0 0 10px;">💬 종합 피드백</h4>
        <p style="color: #cdd6f4; line-height: 1.6;">{result.get('overall_feedback', '')}</p>
        {'<p style="color:#89dceb;"><b>발음 팁:</b> ' + result.get('pronunciation_tips','') + '</p>' if result.get('pronunciation_tips') else ''}
      </div>
    </div>"""

    display(HTML(html))

print('✅ 모든 함수 정의 완료!')

✅ 모든 함수 정의 완료!


In [ ]:
# 영어 음성 목록 확인
async def list_english_voices():
    voices = await edge_tts.list_voices()
    en_voices = [v for v in voices if v['Locale'].startswith('en-')]
    print('🎙️ 사용 가능한 영어 TTS 음성:')
    print(f'{'이름':<35} {'성별':<8} {'로케일'}')
    print('-' * 60)
    for v in en_voices[:15]:  # 상위 15개만 출력
        print(f"{v['ShortName']:<35} {v['Gender']:<8} {v['Locale']}")

await list_english_voices()

🎙️ 사용 가능한 영어 TTS 음성:
이름                                  성별       로케일
------------------------------------------------------------
en-AU-WilliamMultilingualNeural     Male     en-AU
en-AU-NatashaNeural                 Female   en-AU
en-CA-ClaraNeural                   Female   en-CA
en-CA-LiamNeural                    Male     en-CA
en-HK-YanNeural                     Female   en-HK
en-HK-SamNeural                     Male     en-HK
en-IN-NeerjaExpressiveNeural        Female   en-IN
en-IN-NeerjaNeural                  Female   en-IN
en-IN-PrabhatNeural                 Male     en-IN
en-IE-ConnorNeural                  Male     en-IE
en-IE-EmilyNeural                   Female   en-IE
en-KE-AsiliaNeural                  Female   en-KE
en-KE-ChilembaNeural                Male     en-KE
en-NZ-MitchellNeural                Male     en-NZ
en-NZ-MollyNeural                   Female   en-NZ


In [ ]:
!pip install -q nest_asyncio
import nest_asyncio
nest_asyncio.apply()  # Colab 이벤트 루프 충돌 해결

In [ ]:
# ── 테스트용 예문들 (실제 학습자가 자주 틀리는 패턴) ──────────────────
test_sentences = [
    "I am very boring in this class.",           # boring vs bored
    "Yesterday I go to the store and buyed milk.", # 시제 오류
    "She don't know what to do.",                # 주어-동사 불일치
    "I want to discuss about the problem.",      # 불필요한 전치사
    "He is more taller than his brother.",       # 비교급 오류
]

# 테스트할 문장 선택 (0~4)
TEST_INDEX = 0
test_text = test_sentences[TEST_INDEX]

print('=' * 60)
print('🧪 텍스트 → LLM 교정 → TTS 파이프라인 테스트')
print('=' * 60)
print(f'\n입력 문장: "{test_text}"\n')

# 1. LLM 교정
correction = correct_english(test_text)

# 2. 결과 표시
display_correction_result(correction)

# 3. TTS: 교정된 문장을 음성으로
corrected_text = correction.get('corrected', test_text)
audio_path = text_to_speech(
    corrected_text,
    output_path='/content/corrected_output.mp3',
    voice='en-US-JennyNeural'
)

print('\n🔊 교정된 문장 음성 재생:')
display(Audio(audio_path, autoplay=False))

🧪 텍스트 → LLM 교정 → TTS 파이프라인 테스트

입력 문장: "I am very boring in this class."


🤖 LLM 교정 중...
   ⏱️  처리 시간: 0.52초
   📊 자연스러움 점수: 4/10



🔊 TTS 생성 중: "I find this class very boring."
   ✅ 음성 파일 저장: /content/corrected_output.mp3
   ⏱️  처리 시간: 0.33초

🔊 교정된 문장 음성 재생:


In [ ]:
# ── 테스트용 음성 파일 생성 (실제 환경에선 마이크 녹음 또는 파일 업로드) ─
# 먼저 잘못된 영어 문장을 TTS로 생성해서 STT 테스트에 사용합니다

sample_wrong_english = "I am very boring in this class and I don't understood the teacher."

print('🎭 STT 테스트용 샘플 음성 생성 중...')
sample_audio_path = text_to_speech(
    sample_wrong_english,
    output_path='/content/sample_input.mp3',
    voice='en-US-GuyNeural'   # 남성 음성으로 생성
)

print('\n📻 생성된 샘플 음성 (STT 입력):')
display(Audio(sample_audio_path, autoplay=False))
print(f'\n📝 원래 텍스트: "{sample_wrong_english}"')

🎭 STT 테스트용 샘플 음성 생성 중...

🔊 TTS 생성 중: "I am very boring in this class and I don't underst..."
   ✅ 음성 파일 저장: /content/sample_input.mp3
   ⏱️  처리 시간: 0.32초

📻 생성된 샘플 음성 (STT 입력):



📝 원래 텍스트: "I am very boring in this class and I don't understood the teacher."


In [ ]:
# ── 전체 파이프라인: STT → LLM → TTS ─────────────────────────────────

def full_pipeline(audio_path: str, tts_voice: str = 'en-US-JennyNeural') -> dict:
    """
    전체 파이프라인을 실행합니다.
    1. STT: 음성 → 텍스트
    2. LLM: 텍스트 교정
    3. TTS: 교정된 텍스트 → 음성
    """
    total_start = time.time()
    print('\n' + '═' * 60)
    print('🚀 전체 파이프라인 시작')
    print('═' * 60)

    # STEP 1: STT
    print('\n[1/3] 🎙️  STT 처리')
    stt_result = transcribe_audio(audio_path)
    transcribed_text = stt_result['text']

    if not transcribed_text:
        print('❌ 음성 인식 실패: 텍스트가 비어있습니다.')
        return {}

    # STEP 2: LLM 교정
    print('\n[2/3] 🤖 LLM 교정')
    correction = correct_english(transcribed_text)

    # STEP 3: TTS
    print('\n[3/3] 🔊 TTS 생성')
    corrected_text = correction.get('corrected', transcribed_text)
    output_audio = text_to_speech(
        corrected_text,
        output_path='/content/pipeline_output.mp3',
        voice=tts_voice
    )

    total_elapsed = time.time() - total_start
    print(f'\n✅ 전체 파이프라인 완료! 총 소요시간: {total_elapsed:.2f}초')

    return {
        'stt': stt_result,
        'correction': correction,
        'output_audio': output_audio
    }


# ── 파이프라인 실행 ───────────────────────────────────────────────────
result = full_pipeline('/content/sample_input.mp3')

print('\n' + '═' * 60)
print('📋 교정 결과')
print('═' * 60)
display_correction_result(result['correction'])

print('\n🔊 교정된 문장 음성:')
display(Audio(result['output_audio'], autoplay=False))


════════════════════════════════════════════════════════════
🚀 전체 파이프라인 시작
════════════════════════════════════════════════════════════

[1/3] 🎙️  STT 처리
🎙️  STT 처리 중: /content/sample_input.mp3
   📝 인식된 텍스트: "I am very boring in this class and I don't understand the teacher."
   🌐 감지된 언어: en
   ⏱️  처리 시간: 4.53초

[2/3] 🤖 LLM 교정

🤖 LLM 교정 중...
   ⏱️  처리 시간: 0.39초
   📊 자연스러움 점수: 6/10

[3/3] 🔊 TTS 생성

🔊 TTS 생성 중: "I find this class very boring and I don't understa..."
   ✅ 음성 파일 저장: /content/pipeline_output.mp3
   ⏱️  처리 시간: 0.35초

✅ 전체 파이프라인 완료! 총 소요시간: 5.27초

════════════════════════════════════════════════════════════
📋 교정 결과
════════════════════════════════════════════════════════════



🔊 교정된 문장 음성:


In [ ]:
# 여러 문장을 연속으로 교정하는 연습 세션

practice_sentences = [
    "I have went to school yesterday.",
    "She suggested me to try the new restaurant.",
    "I look forward to meet you.",
    "Despite of the rain, we continued playing.",
    "He explained me the situation.",
]

print('📚 영어 회화 연습 세션')
print('=' * 60)

session_results = []

for i, sentence in enumerate(practice_sentences, 1):
    print(f'\n[{i}/{len(practice_sentences)}] 입력: "{sentence}"')
    correction = correct_english(sentence)
    session_results.append(correction)

    score = correction.get('score', 0)
    corrected = correction.get('corrected', '')
    is_ok = correction.get('is_correct', False)

    status = '✅' if is_ok else '❌'
    print(f'   {status} 교정: "{corrected}" (점수: {score}/10)')

# 세션 요약
avg_score = sum(r.get('score', 0) for r in session_results) / len(session_results)
print('\n' + '=' * 60)
print(f'📊 세션 요약')
print(f'   평균 점수: {avg_score:.1f}/10')
print(f'   처리 문장: {len(session_results)}개')
print('=' * 60)

# 가장 낮은 점수 문장에 대한 TTS 교정 출력
lowest = min(session_results, key=lambda x: x.get('score', 10))
print(f'\n🎯 가장 많은 교정이 필요한 문장:')
display_correction_result(lowest)

audio_path = text_to_speech(lowest.get('corrected', ''), '/content/session_best.mp3')
display(Audio(audio_path, autoplay=False))

📚 영어 회화 연습 세션

[1/5] 입력: "I have went to school yesterday."

🤖 LLM 교정 중...
   ⏱️  처리 시간: 0.50초
   📊 자연스러움 점수: 6/10
   ❌ 교정: "I went to school yesterday." (점수: 6/10)

[2/5] 입력: "She suggested me to try the new restaurant."

🤖 LLM 교정 중...
   ⏱️  처리 시간: 0.42초
   📊 자연스러움 점수: 6/10
   ❌ 교정: "She suggested that I try the new restaurant." (점수: 6/10)

[3/5] 입력: "I look forward to meet you."

🤖 LLM 교정 중...
   ⏱️  처리 시간: 0.45초
   📊 자연스러움 점수: 6/10
   ❌ 교정: "I look forward to meeting you" (점수: 6/10)

[4/5] 입력: "Despite of the rain, we continued playing."

🤖 LLM 교정 중...
   ⏱️  처리 시간: 0.43초
   📊 자연스러움 점수: 8/10
   ❌ 교정: "Despite the rain, we continued playing." (점수: 8/10)

[5/5] 입력: "He explained me the situation."

🤖 LLM 교정 중...
   ⏱️  처리 시간: 0.46초
   📊 자연스러움 점수: 6/10
   ❌ 교정: "He explained the situation to me." (점수: 6/10)

📊 세션 요약
   평균 점수: 6.4/10
   처리 문장: 5개

🎯 가장 많은 교정이 필요한 문장:



🔊 TTS 생성 중: "I went to school yesterday."
   ✅ 음성 파일 저장: /content/session_best.mp3
   ⏱️  처리 시간: 0.24초
